# Tunnel analysis from Google Drive

Keep editing and testing in Codespaces. Commit and push when ready; this notebook downloads a fixed Git commit and runs the existing package. It does not contain a second segmentation implementation.

Use **Runtime → Change runtime type → CPU** (no accelerator). Run the cells in order, or select **Runtime → Run all** after completing the fields. Your first execution authorizes Drive access. No GitHub token is needed for the public repository.

Results are provisional biological measurements. This launcher does not change segmentation settings or validate their biological accuracy.


In [ ]:
#@title 1. Choose folders and code
INPUT_FOLDER = "/content/drive/MyDrive/Burrowing experiment" #@param {type:"string"}
RESULTS_PARENT = "/content/drive/MyDrive/Burrowing experiment/Results" #@param {type:"string"}
RESUME_RUN = "" #@param {type:"string"}
CODE_REVISION = "main" #@param {type:"string"}
RUN_SENSITIVITY = True #@param {type:"boolean"}

# Leave RESUME_RUN empty for a new dated run. To resume, paste the full run
# directory printed by cell 3, including its run_... name. A resume uses the
# saved input folder, commit, configuration, and sensitivity settings instead
# of the new-run fields above. CODE_REVISION may be a branch, tag, or commit.


In [ ]:
#@title 2. Connect Google Drive
import sys
if sys.version_info < (3, 11):
    raise RuntimeError("This project requires Python 3.11 or newer.")
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
#@title 3. Download the selected commit and prepare an isolated environment
from datetime import datetime, timezone
from pathlib import Path
import json
import platform
import re
import subprocess
import tempfile
import uuid
import zipfile

REPOSITORY = "https://github.com/prateekdwv/tunnel-analysis.git"
DRIVE_ROOT = Path("/content/drive").resolve()
LOCAL_ROOT = Path("/content/tunnel-analysis-runtime")
LOCAL_ROOT.mkdir(exist_ok=True)


def read_resume_metadata(run):
    state = run.parent / ".tunnel-checkpoints" / run.name / "state.json"
    if state.exists():
        metadata = json.loads(state.read_text())
    else:
        with zipfile.ZipFile(run / "audit.zip") as archive:
            metadata = json.loads(archive.read("run.json"))["batch"]
    if metadata.get("format_version") != 1:
        raise ValueError("Unsupported batch format. Use a run created by the batch command.")
    return metadata


def checkout_revision(repository, revision, parent):
    if not re.fullmatch(r"[A-Za-z0-9_][A-Za-z0-9_./-]*", revision):
        raise ValueError("Enter a Git branch, tag, or commit, without spaces or command options.")
    checkout = Path(tempfile.mkdtemp(prefix="code-", dir=parent))
    subprocess.run(["git", "init", "--quiet", str(checkout)], check=True)
    subprocess.run(["git", "-C", str(checkout), "remote", "add", "origin", repository], check=True)
    subprocess.run(["git", "-C", str(checkout), "fetch", "--quiet", "--depth=1", "origin", revision], check=True)
    commit = subprocess.check_output(
        ["git", "-C", str(checkout), "rev-parse", "FETCH_HEAD^{commit}"], text=True).strip()
    if not re.fullmatch(r"[0-9a-f]{40}", commit):
        raise ValueError("Could not resolve an exact Git commit.")
    subprocess.run(["git", "-C", str(checkout), "checkout", "--quiet", "--detach", commit], check=True)
    return checkout, commit


resume_metadata = None
if RESUME_RUN.strip():
    RUN_DIRECTORY = Path(RESUME_RUN).expanduser().resolve()
    RUN_DIRECTORY.relative_to(DRIVE_ROOT)
    resume_metadata = read_resume_metadata(RUN_DIRECTORY)
    identity = resume_metadata["identity"]
    if not identity["code"]["git_commit"] or identity["code"]["git_dirty"]:
        raise ValueError("This run used unidentified or uncommitted code. Resume it in its original local environment.")
    if identity["environment"]["python"] != platform.python_version():
        raise ValueError("Python differs from the saved run. Use the original Python version or start a new run.")
    revision = identity["code"]["git_commit"]
    SOURCE_DIRECTORY = Path(resume_metadata["input_directory"])
else:
    revision = CODE_REVISION.strip()
    SOURCE_DIRECTORY = Path(INPUT_FOLDER).expanduser().resolve()
    parent = Path(RESULTS_PARENT).expanduser().resolve()
    parent.relative_to(DRIVE_ROOT)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    RUN_DIRECTORY = parent / ("run_" + stamp + "_" + uuid.uuid4().hex[:8])
SOURCE_DIRECTORY.relative_to(DRIVE_ROOT)
if not SOURCE_DIRECTORY.is_dir():
    raise ValueError("Input folder does not exist. Copy its mounted path from Colab's Files sidebar.")

CHECKOUT, COMMIT = checkout_revision(REPOSITORY, revision, LOCAL_ROOT)
if not (CHECKOUT / "src/tunnel_analysis/batch.py").exists():
    raise RuntimeError("This Git revision has no batch launcher. Commit and push the integration from Codespaces first.")
ENVIRONMENT = Path(tempfile.mkdtemp(prefix="env-", dir=LOCAL_ROOT))
try:
    subprocess.run([sys.executable, "-m", "venv", str(ENVIRONMENT)], check=True)
except subprocess.CalledProcessError:
    # Some hosted Python images do not include ensurepip for stdlib venv.
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "virtualenv"], check=True)
    subprocess.run([sys.executable, "-m", "virtualenv", str(ENVIRONMENT)], check=True)
PYTHON = ENVIRONMENT / "bin/python"
if resume_metadata:
    dependencies = resume_metadata["identity"]["environment"]["dependencies"]
    pins = []
    for name, version in sorted(dependencies.items()):
        if name == "tunnel-analysis":
            continue
        if not re.fullmatch(r"[A-Za-z0-9_.-]+", name) or not re.fullmatch(r"[A-Za-z0-9.!+_-]+", version):
            raise ValueError("Unsupported dependency entry in saved run.")
        pins.append(name + "==" + version)
    subprocess.run([str(PYTHON), "-m", "pip", "install", "--quiet", *pins], check=True)
    install_options = ["--no-deps"]
else:
    subprocess.run([str(PYTHON), "-m", "pip", "install", "--quiet", "setuptools>=68", "wheel"], check=True)
    install_options = []
subprocess.run([str(PYTHON), "-m", "pip", "install", "--quiet", "--no-build-isolation",
                *install_options, "--editable", str(CHECKOUT)], check=True)
print("Fixed Git commit:", COMMIT)
print("Input folder:", SOURCE_DIRECTORY)
print("Run directory (save this to resume):", RUN_DIRECTORY)


In [ ]:
#@title 4. Run or resume the analysis
SCRATCH = LOCAL_ROOT / "scratch"
SCRATCH.mkdir(exist_ok=True)
command = [str(PYTHON), "-m", "tunnel_analysis", "batch", str(SOURCE_DIRECTORY),
           "--output", str(RUN_DIRECTORY), "--scratch-dir", str(SCRATCH)]
if resume_metadata:
    command.append("--resume")
else:
    command += ["--source-root", str(DRIVE_ROOT), "--config", str(CHECKOUT / "config.json")]
    if not RUN_SENSITIVITY:
        command.append("--no-sensitivity")
completed = subprocess.run(command, check=False)
if completed.returncode:
    raise RuntimeError(
        "The command reported an interruption, error, or recorded image failures. "
        "Read the messages above. Completed results/checkpoints are preserved at "
        + str(RUN_DIRECTORY) + ". Do not delete checkpoints if you want to resume.")
print("Verified results saved:", RUN_DIRECTORY)


In [ ]:
#@title 5. View results
from IPython.display import HTML, Image, display
import html
import csv

with (RUN_DIRECTORY / "results.csv").open(newline="") as stream:
    rows = list(csv.DictReader(stream))
print("Results folder:", RUN_DIRECTORY)
display(HTML('<a href="https://drive.google.com/drive/my-drive" target="_blank">Open Google Drive</a>'))
for row in rows[:20]:
    print(row["image"], "|", row["tunnel_area_px"], "pixels |", row["status"], "|", row["notes"])
if len(rows) > 20:
    print("Showing the first 20 rows; see results.csv for all measurements.")
for path in sorted(RUN_DIRECTORY.glob("*_overlay.png"))[:3]:
    print(path.name)
    display(Image(filename=str(path), width=650))
print("All overlays, results.csv, and audit.zip remain in the results folder.")


## Resume and later code changes

- If Colab disconnects, open a new session, enter the saved full **RESUME_RUN** path, and run the cells again. The saved commit and dependency versions are restored. Completed images are reused after checksum verification.
- New images, changed image contents, changed settings, or a different code version require a **new run**. Leave RESUME_RUN empty and choose the desired code revision. The input directory is scanned directly, without subfolders.
- The hidden `.tunnel-checkpoints/<run-name>/` directory beside the final run stores recovery data. Keep it until the run finishes. It is removed after final outputs have been verified. Use only one active session for a given run.
- Unreadable TIFF formats and failed region detections are recorded as failed images; other images continue. A completed batch with failures returns an error status so you review the CSV. Corrected images belong in a new run.
- Keep editing and testing in Codespaces. Push your changes only when ready. Already running analyses keep their chosen commit.
- The notebook is manually started and does not watch for new uploads. Colab resources and session duration are limited. A notebook-level failure may leave local scratch until the runtime is reset; durable checkpoints remain on Drive.

[Project README](https://github.com/prateekdwv/tunnel-analysis/blob/main/README.md) · [Colab runtime and Drive documentation](https://research.google.com/colaboratory/faq.html)
